# ICSA Tutorial: Instance-Configuration Space Analysis

This notebook is an introductory tutorial for **Instance-Configuration Space Analysis (ICSA)**,
the methodology and toolkit in this repository.

**Topics:**

1. How to use the interactive toolkit. This is a dual-space visualisation for exploring how algorithm
   performance depends jointly on *instance features* and *parameter settings*.
2. How the projection model works and how to train it yourself, including the difference between the
   two projection modes (`overlay` and `concatenate`).
3. The structure of the input metadata, so you can apply ICSA to your own algorithm and
   problem instances.

**Prerequisites:**

- Install the dependencies as described in the [README](README.md) (Python 3.10+, `pip install -r requirements.txt`).
- Launch Jupyter from the repository root. Every data path in this repository is relative to it.

In [ ]:
import pathlib

assert pathlib.Path('icsa.py').exists() and pathlib.Path('analysis').exists(), \
    "Please run this notebook from the repository root: all data paths are relative to it."

import numpy as np
import pandas as pd
import panel as pn

from icsa import ICSA, set_seeds
from classification_icsa import load_knn_metadata, load_sgd_metadata, MCC

pn.extension('plotly')
pn.config.theme = 'dark'
set_seeds(0)

SHOW_APP = True

## 1. Background: from instance spaces to instance-configuration spaces

### 1.1 Instance Space Analysis

The performance of an algorithm depends on the problem instance it is applied to. *Instance Space
Analysis* (ISA) makes that dependence visible: each instance (e.g., classification dataset) is described by a vector of *features*
(for classification datasets these are "meta-features" such as class entropy or boundary complexity),
and all instances are projected into a 2D *instance space* so that trends in features and in algorithm
performance can be inspected visually. Regions where an algorithm performs well or poorly become visible
areas of the plot, and can be explained in terms of the features that vary across the space. ISA is a
well-established methodology with a large library of analyses at
[MATILDA](https://matilda.unimelb.edu.au/).

### 1.2 What ICSA adds

ISA treats each algorithm as a black box. But an algorithm's behaviour also depends on its
*parameter configuration*. Treating every configuration as a separate "algorithm" does not scale
and ignores that similar configurations behave similarly.

ICSA extends ISA with a *second 2D space for configurations*, learned *jointly* with the instance
space. A multi-head neural network is trained with three components:

- an *instance encoder*: feature vector -> 2D instance point $\mathbf{z} = (z_1, z_2)$,
  with a decoder reconstructing the features from $\mathbf{z}$;
- a *configuration encoder*: parameter vector -> 2D configuration point $\mathbf{c} = (c_1, c_2)$;
- a *performance head*: predicts the performance $\hat{y}(\mathbf{z}, \mathbf{c})$ of a configuration
  on an instance from the two projected points.

The training loss combines these (defaults from `projector.Losses`):

$$\mathcal{L} = 10.0 \cdot \text{MSE}(y, \hat{y}) \; + \; 1.0 \cdot \text{MSE}(F, \hat{F})$$

The *performance loss* forces the layout of both spaces to be meaningful in terms of performance. Nearby
instances have similar performance patterns over the configurations, and vice versa. The *feature
reconstruction loss* keeps the instance space *interpretable in feature terms*, as in ISA: feature
values vary smoothly (ideally linearly) across the space.

### 1.3 The two projection models

ICSA supports two forms of the performance head, selected by the `mode` argument of `ICSA(...)`:

| | `mode='overlay'` (OVERLAYING) | `mode='concatenate'` (CONCATENATE) |
|---|---|---|
| Performance prediction | fixed function $\hat{y} = 1/(1 + \lVert\mathbf{c}-\mathbf{z}\rVert^2)$ | feedforward network on $(z_1, z_2, c_1, c_2)$ |
| Relationship between the spaces | *shared coordinates*: a configuration performs best on the instances it sits *on top of* | separate spaces, related only through the learned network |
| Reading the plot | distance means performance | inspect via selections and recolouring |
| Instance encoder default | 3 hidden layers (nonlinear) | *0 hidden layers (linear)*: feature trends are linear across the space |
| Performance preprocessing default | min-max scaled per instance (relative performance) | unscaled (absolute performance) |
| Strengths | reveals coarse structure: well-separated groups of instances and configurations | better performance prediction, finer-grained, more nuanced insights |

We will use both in this tutorial and compare them in Section 4.

### 1.4 How to read the interactive app

The toolkit shows both 2D spaces in one figure: *circles* are instances (axes $Z_1, Z_2$) and
*triangles* are configurations (axes $C_1, C_2$). By default, each point is coloured by its
*aggregate performance*, the mean performance of an instance over all configurations (Viridis
colour scale), or of a configuration over all instances (Plasma colour scale):

![Aggregate view of the KNN overlaying projection](Figures/KNN_Overlaying_AggAgg_1.png)

The key interactivity is as follows: *lasso-select* points in one space, and the other space is recoloured by
aggregate performance *over your selection*. Select a group of instances and you see which
configurations are good for those instances specifically. Select configurations and you see which
instances they find easy or hard. Side panels provide notched-median plots and box plots of performance
against parameter values, a table of selected configurations, and a feature percentile-rank heatmap for
selected instances. Section 3 walks through all of this.

### 1.5 The case study bundled with this repository

All data in `analysis/classification/` comes from the paper's case study:

- *235 classification datasets* (UCI/KEEL/DCol, as used in MATILDA's classification ISA), each
  described by *10 meta-features*: `Max_Normalized_Entropy_attributes`,
  `Normalized_Entropy_Class_Attribute`, `Mean_Mutual_Information_Attribute_Class`,
  `ErrorRate_Decision_Node`, `WeightedDist_StdDev`, `Max_Feature_Efficiency_F3`,
  `Collective_Feature_Efficiency_F4`, `Training_Error_Linear_Classifier_L2`,
  `Fraction_Points_Class_Boundary_N1`, `Nonlinearity_Nearest_Neighbor_Classifier_N4`.
- *Two scikit-learn classifiers* with sampled parameter configurations: *KNN* (168 configurations:
  a full factorial over its parameter levels) and *SGDClassifier* (666 configurations: a strength-3
  covering array over a much larger parameter space).
- *Performance*: Matthews Correlation Coefficient (MCC, higher is better), computed from 5-fold
  stratified cross-validation on every (dataset, configuration) pair.

## 2. The metadata: the inputs to ICSA

What ICSA needs is *three tables plus three small dicts for plotting*, bundled in a
`SimpleNamespace`. As a case study, we will load the KNN metadata
and inspect each aspect of the methodology.

In [ ]:
metadata = load_knn_metadata('analysis/classification/knn/', confusion_function=MCC)

### 2.1 `feature_data`: one row per instance

A numeric DataFrame, *indexed by instance ID* (e.g., dataset names), one column per feature.
These features were already preprocessed. ICSA can scale features
(`feature_preprocessing='encode and scale'`), but preprocessing is up to you.

In [ ]:
print(metadata.feature_data.shape)
metadata.feature_data.head(3)

### 2.2 `parameter_data`: one row per configuration

A DataFrame, *indexed by configuration ID* (e.g., integers 0 to 167), one column per parameter.
Categorical parameters are strings, and numerical parameters are numbers. ICSA one-hot encodes and
standardises them automatically (`parameter_preprocessing='encode and scale'`, the default).

In [ ]:
print(metadata.parameter_data.shape)
metadata.parameter_data.head(3)

### 2.3 `performance_data`: one row per (instance, configuration) pair

A *long-format* DataFrame whose *first three columns* are, in order:
*(instance ID, configuration ID, performance value)*. The IDs must match the indexes of
`feature_data` and `parameter_data`, and *every pair must be present*. ICSA pivots this into a
complete 235 × 168 performance matrix internally.

In [ ]:
print(metadata.performance_data.shape)   # 235 instances x 168 configurations = 39,480 rows
metadata.performance_data.head(3)

### 2.4 The three plotting extras

The interactive app additionally needs, for display purposes only:

- *`parameter_design`*: a *string-typed* copy of `parameter_data`. It is what appears in the
  configuration table and on the box-plot axes (e.g. numeric levels shown as interval labels).
- *`parameter_domain`*: a dict mapping each parameter to its *ordered* list of levels. This
  controls the category order on plot axes.
- *`parameter_type`*: a dict mapping each parameter to `'c'` (categorical), `'o'` (ordinal) or
  `'r'` (real). This controls how the parameter is colour-mapped over the configuration space.

In [ ]:
print(metadata.parameter_domain)
print(metadata.parameter_type)
metadata.parameter_design.head(3)

### 2.5 The minimal contract (for your own data)

| Field | Type | Requirement |
|---|---|---|
| `feature_data` | DataFrame (n_instances × n_features), numeric | index = instance IDs |
| `parameter_data` | DataFrame (n_configs × n_params) | index = configuration IDs, categoricals as `str`, numericals as numbers (encoding/scaling is automatic) |
| `performance_data` | DataFrame, long format, *first three columns* = (instance ID, config ID, value) | IDs match the two indexes above, the (instance × config) matrix must be *complete* |
| `parameter_design` | DataFrame (n_configs × n_params), all `str` | display copy of `parameter_data` for tables and box plots |
| `parameter_domain` | dict: parameter -> ordered list of levels | plot category ordering |
| `parameter_type` | dict: parameter -> `'c'` / `'o'` / `'r'`, categorical, ordinal, or real | plot colour mapping |

> Note: *The configurations do not need to come from a covering array (like the paper).* The paper sampled SGDClassifier's
> large parameter space with a covering array (via the CAGen tool) to cover all low-order parameter
> interactions efficiently. However, that is just one sampling strategy. A full factorial (as used for KNN
> here), a random or Latin-hypercube sample, or even the evaluation history of a configurator all
> work, as long as every sampled configuration has been evaluated on every instance, or if their performance is estimated via an Empirical Performance Model (EPM).

Building the metadata by hand is straightforward (illustrative snippet, not executed here):

```python
from types import SimpleNamespace

parameter_data = pd.DataFrame(list_of_config_dicts)           # any sample of configurations
parameter_design = parameter_data.astype(str)
metadata = SimpleNamespace(
    feature_data=feature_data,
    parameter_data=parameter_data,
    performance_data=performance_data,                        # long: (instance_id, config_id, value)
    parameter_design=parameter_design,
    parameter_domain={p: sorted(parameter_design[p].unique()) for p in parameter_design.columns},
    parameter_type={p: 'c' for p in parameter_design.columns},  # or 'o'/'r' where appropriate, 'c'/'o'/'r': categorical/ordinal/real
    parameter_default=None,
)
```

## 3. The interactive toolkit (on precomputed projections)

Trained projections for the case study are included in the repository as `z.npy` (instance points),
`c.npy` (configuration points) and `performance_matrix.npy`, under
`analysis/classification/{knn,sgdclassifier}/mcc/{overlapping,concatenated}/`. Hence, we can explore
the app without training anything.

In `demo.ipynb`, `ICSA(...).fit_show(model_fpath, metadata, save=True)` is used, which
loads the precomputed projection if it exists (otherwise trains one) and then immediately launches the
app. However, in this tutorial we use a helper `build_app` instead.

---

**NOTE**: In its current state, *the UI may not fit all monitor resolutions*. There is still remaining polish to be done.
The temporary workaround is to change zoom in the browser (e.g., by pressing `ctrl` and scrolling). In general, for the current state of the project,
**there still remains much to be done for convenient use and software polish**.

---

In [ ]:
def build_app(mode, model_fpath, metadata, performance_metric_name='MCC'):
    """Load a precomputed projection from `model_fpath` and return (analysis, app).

    Mirrors `ICSA.fit_show(...)` -> `ICSA.show(...)` (see icsa.py), but returns the Panel
    app instead of launching a server right away. Note: as in `ICSA.show()`, the string-typed
    `parameter_design` is passed both as the plotter's parameter table and as the box-plot design.
    """
    analysis = ICSA('max', mode=mode, performance_metric_name=performance_metric_name)
    assert analysis.is_fitted(model_fpath), f"No precomputed projection found at {model_fpath}"
    analysis.load_fitted_model(model_fpath, metadata.feature_data, metadata.parameter_data)
    app = analysis.plotter.get(
        metadata.feature_data, metadata.parameter_design, analysis._performance_matrix,
        analysis._z, c=analysis._c,
        parameter_design=metadata.parameter_design,
        parameter_domain=metadata.parameter_domain,
        parameter_type=metadata.parameter_type,
    )
    return analysis, app

In [ ]:
knn_overlay, knn_overlay_app = build_app('overlay', 'analysis/classification/knn/mcc/overlapping/', metadata)

if SHOW_APP:
    knn_overlay_server = knn_overlay_app.show(threaded=True)   # opens the app in a new browser tab

`threaded=True` keeps the notebook responsive while the app runs. Call `knn_overlay_server.stop()`
when you are done with it.

**Introduction to the interface:**

- *Main figure (right):* circles = instances on axes $Z_1/Z_2$, triangles = configurations on
  $C_1/C_2$. In *overlay* mode the two coordinate systems coincide. Drag mode is *lasso* by default.
  Double-click to reset the view. Hover any point for its ID and value.
- *`Plot (Instance Space)` / `Plot (Configuration Space)` dropdowns (top-left):* choose what colours
  each space: `Aggregate Performance` (default), any instance feature, or any parameter.
- *`Focus:` toggle:* chooses which space your lasso selects points in (the *focused* space is drawn
  with larger markers).
- *`Reset Selection:` buttons:* clear the instance or configuration selection.
- *Tabs (bottom-left):* `Notches` (median ± interval plots), `Boxplots`, a table of the selected
  configurations with their aggregate performance, and a percentile-rank feature heatmap of the selected
  instances. The `X-Axis`, `Colour` and `Facet` dropdowns control the Notches/Boxplots views.

The follow exercises end with a reference figure from the paper so you can check what you
should be seeing (your point layout may be mirrored or rotated, but only relative structure matters).

---

### Exercise 1: Read the aggregate view

1. Look at the instance circles (coloured by `Aggregate Performance`, i.e. mean MCC over all 168
   configurations). Find the two distinct instance clusters: an upper group and a lower group. The
   lower group is noticeably brighter (easier).
2. Make sure `Focus:` is `Instance Space`, then *lasso the upper instance group*. The configuration
   triangles recolour to their aggregate MCC *over just those instances*. Note the values on the
   colour bar drop, and a performance gradient appears across the configuration cloud.
3. Press `Reset Selection: Instances`, then lasso the *lower group* and compare.

![Aggregate performances after selecting the upper instance group](Figures/KNN_Overlaying_AggAgg_1.png)

### Exercise 2: `n_neighbors` and class-boundary complexity

1. Set `Plot (Instance Space)` to `Fraction_Points_Class_Boundary_N1` (the fraction of points on the
   class boundary: high values mean noisy/complex boundaries). Note the upper instance group has high
   values.
2. Set `Plot (Configuration Space)` to `n_neighbors`.
3. Observe: configurations with *larger `n_neighbors` sit closer to the high-boundary instances*
   (in overlay mode, proximity = predicted good performance). Interpretation: more neighbours smooth
   KNN's decision boundary, which pays off on noisy boundaries, while low-`n_neighbors`
   configurations sit near the easier, low-boundary instances.

![FPCB over instance space and n_neighbors over configuration space](Figures/KNN_Overlaying_FPCBNN.png)

### Exercise 3: The distance metric

1. Set `Plot (Configuration Space)` to `metric` and `Plot (Instance Space)` to
   `Aggregate Performance`.
2. Find the isolated configuration group far to the right: it is entirely `chebyshev`. Its distance
   from *all* instances means it performs poorly everywhere. Hence, a global weakness, not an instance-dependent one.
3. The other metrics mix within the left configuration group. Lasso a few sub-regions (set `Focus:` to
   `Configuration Space`) and watch the instance space recolour to show which instances those
   configurations find easy or hard.

![ERDN over instance space and metric over configuration space](Figures/KNN_Overlaying_ERDNMetric.png)

### Exercise 4: Lasso + notched medians

1. Set `Focus:` back to `Instance Space` and lasso the *upper* instance group.
2. Open the `Notches` tab. Set `X-Axis` to `n_neighbors` and `Colour` to `weights`.
3. Read the plot: each marker is the median aggregate MCC of the configurations with that parameter
   value, over your selected instances. The vertical interval is a confidence notch
   (non-overlapping intervals roughly mean significantly different medians). For these instances, performance is
   robust to high `n_neighbors`, and `distance` weighting beats `uniform` almost everywhere.
4. Reset and repeat for the *lower* instance group: now performance *decays* clearly beyond
   `n_neighbors` = 8. The best `n_neighbors` depends on where the instances live in feature space.

![Notched medians, upper instance group](Figures/KNN_Overlaying_Upper_Instances_Notches.png)
![Notched medians, lower instance group](Figures/KNN_Overlaying_Lower_Instances_Notches.png)

## 4. Overlay vs concatenate

The same metadata, projected with the CONCATENATE model, is also precomputed. We will open it side by
side with the overlay app:

In [ ]:
knn_concat, knn_concat_app = build_app('concatenate', 'analysis/classification/knn/mcc/concatenated/', metadata)

if SHOW_APP:
    knn_concat_server = knn_concat_app.show(threaded=True)

Things to notice, and when to prefer which model:

- *The spaces are now separate.* Proximity between a triangle and a circle no longer means anything.
  Click on a trace (C or Z) in the upper right to disable the configuration/instance trace.
  The $C_1/C_2$ axes (top/right) are independent of $Z_1/Z_2$ (bottom/left). You explore relationships
  through *selections* instead of overlap.
- *The instance space is a linear projection of the features* (the instance encoder has 0 hidden
  layers in this mode, as in classical ISA), so feature trends across the space are linear and easy to
  read. Try `Plot (Instance Space)` -> `Fraction_Points_Class_Boundary_N1`.
- *Finer structure appears.* Set `Plot (Configuration Space)` to `metric`: the `cosine`
  configurations now form their own clearly separated group (in the overlay projection they were mixed
  in). Set `X-axis` to `metric` on the left panel. Then, Lasso high-FPCB instances and then low-FPCB instances and watch the `cosine` group flip from
  strong to weak: a nuance the overlay projection did not separate:

![FPCB over the concatenate instance space](Figures/KNN_Concatenated_FPCB.png)
![metric over the concatenate configuration space](Figures/KNN_Concatenated_Configuration_Metric.png)

*Rules of thumb* (from the paper's comparison):

- `overlay`: intuitive distance semantics, good at revealing *coarse, large-scale structure*
  (well-separated instance groups and configuration groups with very distinct behaviour).
- `concatenate`: a more flexible performance model, hence *better performance prediction for the same
  architecture* and *more nuanced insights*, supports linear, ISA-style instance spaces.

Also note that `mode` silently switches other defaults: the instance encoder depth (3 hidden layers vs
0) and the performance preprocessing (`'minmax scale rows'`, relative performance per instance, vs
`'unscaled'`). Both can be overridden explicitly (Section 5).

### Exercise 5: SGDClassifier `average` parameter depends on feature informativeness

The second case study is SGDClassifier, with 666 configurations over 9 parameters:

In [ ]:
sgd_metadata = load_sgd_metadata('analysis/classification/sgdclassifier/', confusion_function=MCC)
sgd_concat, sgd_concat_app = build_app('concatenate', 'analysis/classification/sgdclassifier/mcc/concatenated/', sgd_metadata)

if SHOW_APP:
    sgd_concat_server = sgd_concat_app.show(threaded=True)

1. Set `Plot (Instance Space)` to `Mean_Mutual_Information_Attribute_Class` (MMIAC: how informative
   the attributes are, individually, about the class). There is a linear trend across the instance
   space.
2. Lasso the *low-MMIAC* instances. Open `Notches`, set `X-Axis` to `average`. Averaging the SGD
   weight updates (`average = True`) wins: with weakly informative attributes, gradient updates are
   noisy, and averaging smooths them.
3. Reset, lasso the *high-MMIAC* instances. The effect reverses: `average = False` is better, as
   smoothing now slows convergence to the optimal weights.
4. Add `Colour` = `learning_rate` (or `penalty`) to see where the interaction holds and where it
   vanishes (e.g., it disappears under the `adaptive` learning-rate schedule).

![average, low MMIAC](Figures/SGD_Concatenated_MMIAC_Low_Av.png)
![average, high MMIAC](Figures/SGD_Concatenated_MMIAC_High_Av.png)


## 5. Training the projection model yourself

`ICSA` exposes ~40 hyperparameters (all documented in the class docstring in [icsa.py](icsa.py)). The
 ones that matter first:

| Argument | Meaning |
|---|---|
| `optimisation_direction` (positional) | `'max'` if higher performance values are better (MCC), `'min'` otherwise (e.g. error rate, runtime) |
| `mode` | `'overlay'` or `'concatenate'` (Section 4) |
| `performance_loss_weight` / `feature_loss_weight` | default: a 10 : 1 trade-off between performance fidelity and feature interpretability of the layout |
| `instance_encoder_num_hidden_layers` | `0` = linear, ISA-style instance space. More layers = more flexible, less interpretable (default depends on `mode`) |
| `performance_metric_name` | axis/colour-bar label only |
| `epochs`, `validation_split` | passed to `fit(...)`. The paper uses 150 epochs |

Training is a normal Keras fit.

In [ ]:
RETRAIN = False   # flip to True to train a projection from scratch

if RETRAIN:
    set_seeds(0)
    my_analysis = ICSA('max', mode='overlay', performance_metric_name='MCC')
    my_analysis.fit(metadata.feature_data, metadata.parameter_data, metadata.performance_data,
                    validation_split=0.1, epochs=150, verbose=1)
    my_app = my_analysis.plotter.get(
        metadata.feature_data, metadata.parameter_design, my_analysis._performance_matrix,
        my_analysis._z, c=my_analysis._c,
        parameter_design=metadata.parameter_design,
        parameter_domain=metadata.parameter_domain,
        parameter_type=metadata.parameter_type,
    )
    if SHOW_APP:
        my_server = my_app.show(threaded=True)

Notes:

- *Results may differ due to potential hardware-level
  nondeterminism, via Keras*, and 2D layouts are only identified up to rotation/reflection.
- *Persisting a model:* `my_analysis.save_fitted_model(fpath)` writes `z.npy`, `c.npy` and
  `performance_matrix.npy` into `fpath`. Save to a *new directory you create yourself* (e.g.
  `analysis/classification/knn/mcc/tutorial_run/`) to not overwrite the precomputed results. `load_fitted_model(fpath, feature_data, parameter_data)` loads them back.
- `ICSA('max', mode='overlay').fit_show(model_fpath, metadata, epochs=150, save=True)`
  does all of the above: loads the model at `model_fpath` if present, otherwise trains, saves and
  launches the app. This is what `demo.ipynb` does.

## 6. Using ICSA on your own algorithm and instances

Everything you need to produce is the metadata contract from Section 2.5:

- [ ] `feature_data`: features/meta-features for your instances (numeric DataFrame, index = instance IDs).
- [ ] `parameter_data`: the sampled configurations (DataFrame, index = configuration IDs).
- [ ] `performance_data`: your algorithm's performance for *every* (instance, configuration) pair,
      long format, columns in the order *(instance ID, configuration ID, value)*.
- [ ] `parameter_design`, `parameter_domain`, `parameter_type`: the three plotting extras
      (`parameter_design = parameter_data.astype(str)` is usually enough to start).

Template (not executed):

```python
from types import SimpleNamespace
from icsa import ICSA

my_metadata = SimpleNamespace(
    feature_data=...,       # DataFrame (n_instances x n_features), numeric, index = instance IDs
    parameter_data=...,     # DataFrame (n_configs x n_params), index = config IDs
    performance_data=...,   # long DataFrame: (instance_id, config_id, value), complete matrix
    parameter_design=...,   # parameter_data.astype(str)
    parameter_domain=...,   # {param: ordered list of levels}
    parameter_type=...,     # {param: 'c' | 'o' | 'r'}
    parameter_default=None, # only used by the bundled sklearn loaders
)

analysis = ICSA('max', mode='overlay', performance_metric_name='my metric')
analysis.fit_show('my_results/', my_metadata, epochs=150, save=True)
```

### 6.1 What the software assumes about your data

Before committing to a new application, check your data against these assumptions and requirements.

**Evaluation data (the performance table)**

1. *Complete evaluation matrix.* Every instance × configuration pair must have been evaluated (or
   estimated, e.g. via an empirical performance model). The projection *method* does not fundamentally
   require this, it trains on batches of evaluations, but the current code does, and it fails
   with an explicit error if any pair is missing or NaN.
2. *One value per pair.* Aggregate repeated measurements (multiple seeds or runs) to a single scalar
   per pair beforehand. Duplicate (instance, configuration) rows are rejected at the pivot step.
3. *The metric should be comparable across instances.* Bounded/normalised metrics such as MCC are
   ideal. The `overlay` model min-max scales per instance (it only sees *relative* performance), but
   `concatenate` trains on unscaled values by default, so an unbounded metric like raw runtime lets
   large instances dominate the loss. Note also that `optimisation_direction` only transforms the data
   in the overlay path. For a "lower is better" metric in concatenate mode, transform the metric
   yourself (negate, log, or normalise per instance).
4. *No missing values anywhere.* There is no imputation or validation for the feature and parameter
   tables; NaNs there flow straight into the network. *Ensure clean inputs*.

**Features (the instance table)**

5. *A feature set must exist for your domain.* For a new problem domain, the biggest practical cost
   is often defining and computing instance features at all. In practice all features must be numeric
   (the percentile-rank table transforms the whole feature table at once).
6. *Feature selection is effectively assumed.* The feature reconstruction loss would ideally not
   include many irrelevant features, so select features first. Parameter selection is *not* assumed
   (there is no parameter reconstruction loss by default), but for the visualisation, parameter
   selection or importance weighting is useful when there are many parameters.

**Parameters (the configuration table)**

7. *Any configuration sample works* (Section 2.5): full factorial, random or Latin-hypercube
   samples, or an evaluation history. Covering arrays, as used in the paper, are an efficiency option whose main payoff
   is balanced coverage of low-order parameter interactions for the boxplot/notch tools.
8. *Discretise parameter values into levels.* The boxplot/notch tools group by level, and point
   colouring matches values against `parameter_domain` by *string equality*. So continuous parameters
   need interval levels, and any value not declared in the domain renders uncoloured.
9. *Conditional parameters follow a convention.* Parameters that are inactive for some configurations
   are marked with an explicit "not applicable" sentinel rather than missing values, plus a small
   per-algorithm rule at encoding time: keep NA as a categorical level for categorical parameters, or
   back-fill the default value for numeric ones (see `load_knn_metadata` / `load_sgd_metadata` in
   [classification_icsa.py](classification_icsa.py) for the two patterns). See `class NA` from [experimental_design.py](experimental_design.py).
10. *At most 24 distinct levels per categorical parameter or feature* (fixed colour palette), but
    plots become unreadable well before that. Group or bin high-cardinality categoricals.

**Scale**

11. *Memory bounds the sweep size.* Training materialises the full long-format data in RAM: the
    feature and parameter vectors are replicated for every (instance × configuration) pair, with no
    streaming. Ask "how big is n × r?" first. In the current state of the software, the scale is
    thousands of instances/configurations, not millions.
12. *The interactive plot has its own limit.* The case studies are ~235 instances × 168-666
    configurations. Tens of thousands of points clutter the scatter and slow the browser-side
    interactivity (selections rebuild per-point state), so subsample or cluster for display.

**Reusing the bundled evaluation pipeline**

13. *The evaluation harness is sklearn-classification-specific.* `ClassificationMetadataGenerator`
    assumes `.arff` instance files with a label column named `class`, with per-instance imputation,
    one-hot encoding and standard scaling, uses stratified k-fold CV, and stores *confusion matrices*
    as the performance record. A metric not derivable from a confusion matrix (runtime, regression
    error, probability-based metrics) means changing the storage format, not just swapping the metric
    function. For other domains, write your own evaluation loop targeting the three-table contract:
    it only has to emit *(instance ID, configuration ID, value)* rows.
14. *Sweep resumability is whole-instance.* An interrupted sweep re-runs the entire interrupted
    instance's configuration set, and the projection cannot start until the sweep is 100% complete.
15. *Covering-array route specifics* (only if you use it): `experimental_design.ParameterSpace`,
    `to_acts()` exports your parameter space for the free
    [CAGen](https://srd.sba-research.org/tools/cagen/) web tool, and `get_parameter_design(...)` /
    `generate_parameter_configurations(...)` turn CAGen's CSV back into sampled configurations (numeric
    levels sampled within their intervals: Latin-hypercube). CAGen is a manual step in the
    browser: parameter names must be plain identifiers, constraints are entered by hand in CAGen's
    constraints field, and you must enable *"Randomize don't-care values"* before exporting. Log-scale
    interval sampling requires strictly positive bounds; the default configuration is automatically
    included in the design.

**Practicalities**

16. Run everything from the repository root (paths are relative). The interactive app needs a live
    Jupyter kernel (it is not preserved in a saved notebook). When scripting around `ICSA.fit`
    directly, call `set_seeds` yourself. Only `fit_show` seeds automatically.

### 6.2 Where to look next

The full pipeline, using the classification-specific evaluation harness, 
(parameter space -> covering array -> classifier runs -> metadata) is demonstrated in
[00EXP_classification_icsa_main.ipynb](00EXP_classification_icsa_main.ipynb) (warning: the
classifier runs take ~35+ minutes on many cores, plus the manual CAGen step). The
[README](README.md) documents the same workflow step by step.

---

### Takeaways

- ICSA jointly projects *instances* and *configurations* into two 2D spaces tied together by a
  performance-prediction head, extending ISA to parameter spaces (Section 1).
- The input is a simple three-table contract, and configurations can be sampled *any way you like*
  (Section 2).
- The interactive toolkit turns the projection into contextual insights: lasso a region of instance
  space, and read off which parameter settings work there and why (Sections 3-4).
- `overlay` shows coarse structure with intuitive distance semantics, whereas `concatenate` is more flexible
  and more nuanced (Section 4).
- Training is one `fit_show(...)` call. Models are cached as three `.npy` files (Section 5).
- Check your data against the assumptions in Section 6.1 before committing to a new application.

In [ ]:
# Optional cleanup: stop any app servers started above.
for _name in ['knn_overlay_server', 'knn_concat_server', 'sgd_concat_server', 'my_server']:
    _server = globals().get(_name)
    if _server is not None:
        _server.stop()